# How bandaid removes the ripple

This notebook explains the reasoning behind `bandaid.ripple`, using one example
file and the same functions as the package, so every claim can be checked
against a number or a figure.

**How to read it.** Every section has three parts:

* **In plain terms**: no signal-processing background needed.
* **In detail**: the technical version, with the equations and the numbers.
* **Why?**: answers to the questions that usually come up at that point.

If you only want to know what the software does, read the next section,
"The algorithm at a glance". The rest builds it up step by step.

---

### The whole story, in plain terms

The stripes are not part of the tissue. They are an electrical hum picked up by
the detector electronics, oscillating about 100,000 times per second. The
microscope draws each image line by line with a mirror that swings back and
forth like a pendulum: fast in the middle of its swing, slow near the turning
points. Recording a steady hum with a pendulum-like sweep turns it into
vertical bands that are wide in the center of the image and narrow at the
edges. The hum happens to repeat almost exactly 50 times per image line, so
every line shows the same band pattern and the bands line up vertically. It is
not *exactly* 50 times, so the pattern slides very slowly from one frame to the
next. Knowing all this, we can write down what the pattern must look like,
measure how strong it is and where it sits in every frame, and subtract exactly
that and nothing else.

### The whole story, in detail

The banding is an additive ripple that is periodic in *time*. The resonant
scanner sweeps the beam with a sinusoidal velocity, so pixels near the image
edges are recorded more slowly than pixels in the center; a temporal ripple
therefore appears as vertical bands that get narrower toward the edges. Its
fundamental frequency is a near-integer multiple (50) of the line rate, which
makes every line start at the same ripple phase, so the bands are vertical and
nearly identical on all rows. The small residual offset from that multiple
makes the pattern crawl slowly across frames and puts a slight phase ramp down
each frame. Once each column is mapped to its acquisition time using the
ScanImage scan geometry, the ripple is an ordinary harmonic series with a
known basis; its per-frame amplitudes and phases are found by linear least
squares on row-averaged column profiles and the fitted model is subtracted
from every pixel.

Data: a 600-frame block of the local example session (MINI2P, 2 kHz resonant
scanner, 256 x 256 pixels, int16).

## The algorithm at a glance

This is what `remove_ripple` does to a file, in order. The questions "what is
fitted, on what data, and how often" are answered explicitly because they are
the ones that matter for judging the method.

**1. Reduce every frame to column averages (per frame, not per movie).**
Each 256 x 256 frame is split into 8 blocks of 32 rows. Within each block the
rows are averaged column by column, so a frame becomes an 8 x 256 array of
column profiles. The average is weighted: pixels that belong to the brightest
5 % of the mean image (cell bodies) get weight zero, so their activity does not
leak into the profiles. This is done for *every* frame of the movie, streaming
through the file in chunks. The fit never sees individual pixels and the movie
is never reshaped into a big frames-by-pixels matrix (that is what PCA does).

**2. Detect the ripple (once per file).**
Three blocks of 128 consecutive frames (start, middle, end) are analysed. The
whole-frame column profile of each frame, minus the block's temporal mean, is
expressed in scan time and its power spectrum is averaged. From that: the
fundamental frequency, its harmonic index $n$ relative to the line rate
(50 here), how many harmonics are visible, the amplitude, and how fast the
pattern crawls. Three thresholds decide whether the file is treated at all.
The result is one frequency $f_0 = n / T_{line}$ for the whole file.

**3. Fit the model (per frame; nothing is averaged across frames).**
For each frame, the 8 profiles (2048 numbers) are explained by 16 unknowns:
the cosine and sine amplitude of harmonics 1 to 8. The basis functions
(cos and sin of $2\pi k f_0 \tau(x)$, with the small per-row phase ramp) are
fixed by the scan geometry, so the problem is linear and is solved by ordinary
least squares. For efficiency the solve is done for windows of 256 frames at
once (one pseudo-inverse per window, 256 right-hand sides), but every frame
gets its own coefficients. Three quantities *are* shared:

* $f_0$, one value per file;
* the crawl rate $\rho$ (cycles per row), one value per 256-frame window,
  tracked from the frame-to-frame phase of the fundamental and smoothed with a
  51-frame median filter;
* the static profile (the image structure that is not ripple): first the
  temporal mean of the 8 profiles over the whole movie, then, after a first
  fit, a running mean of the residual over 1024 frames so slow drift of the
  image does not leak into the fit.

The sequence fit -> update $\rho$ -> update static is run twice, then a final
fit. Frames whose fundamental amplitude jumps more than 5 MAD away from its
rolling median (typically motion) are treated as outliers and get coefficients
interpolated from their neighbours.

**4. Rebuild the pattern at full resolution and subtract it (per pixel).**
For each frame, the 16 coefficients are expanded to the full 256 x 256 grid:
each column gets its phase from $\tau(x)$, each row from the ramp
$\rho\,(r - r_0)$, each harmonic its dwell-time attenuation. The result is
subtracted from the raw frame, rounded and clipped to the input dtype, and
written. This is the second streaming pass over the file.

**5. Check.**
The harmonic lines must be gone from the scan-time spectrum of the residual
profiles, and the temporal-mean image must be unchanged. Both numbers go into
the JSON report next to the cleaned file.

**What is deliberately *not* done.** No pixel is smoothed or blurred. Nothing
is filtered in pixel-frequency space (a notch there cannot follow the changing
stripe width). The static image is used only inside the fit and is never
subtracted from the output. Nothing is averaged over frames except the three
shared quantities above.

**Why this design?**

* *Why column averages instead of all pixels?* The ripple is the same on every
  row, so averaging 32 rows keeps it intact while shrinking pixel noise by
  about a factor of 5.7 and averaging out most cell activity; it also makes
  the fit about thirty times cheaper.
* *Why 8 row blocks instead of one?* One whole-frame profile would be blind to
  the slight phase ramp down the frame. Eight blocks let the fit see it, and
  give an independent estimate of the crawl rate.
* *Why per frame?* Because the phase of the pattern changes from frame to
  frame and its rate of change drifts; a single fit for the whole movie would
  be wrong within a few frames.
* *Why least squares?* Because once the basis is known the problem is linear:
  it has a closed-form, cheap solution that is optimal for Gaussian-like
  noise, and it removes only the pattern that matches the known shape.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from bandaid import ScanGeometry, detect_ripple, remove_ripple
from bandaid.diagnostics import plot_tspace_spectra
from bandaid.io import load_tiff_stack, read_scanimage_metadata
from bandaid.ripple import (
    RowGroups,
    column_profiles,
    fit_ripple,
    line_power_ratios,
    ripple_basis,
    ripple_frames,
    tspace_spectrum,
)

plt.rcParams["figure.dpi"] = 110

data_dir = Path("../data/HK13/HK13_20260814")
path = data_dir / "HK13_20260814_50pw_00002.tif"
frames = range(2000, 2600)

meta = read_scanimage_metadata(path, frames=False)
si = meta["si_params"]
geometry = ScanGeometry.from_si_params(si)
mov_int = load_tiff_stack(path, frames=frames)
mov = mov_int.astype(np.float32)
T, H, W = mov.shape

print(f"{path.name}: {T} frames of {H}x{W} {mov_int.dtype}")
print(f"frame rate {si['SI.hRoiManager.scanFrameRate']:.3f} Hz, line period {geometry.line_period_s*1e3:.3f} ms, "
      f"{geometry.lines_per_frame_period} line periods per frame ({geometry.n_flyback_lines} flyback)")
print(f"scanner {geometry.scanner_frequency_hz:.0f} Hz, bidirectional {geometry.bidirectional}, "
      f"fill fraction spatial {geometry.fill_fraction_spatial:.2f} / temporal {geometry.fill_fraction_temporal:.3f}, "
      f"ADC {geometry.sample_rate_hz/1e6:.0f} MHz")

## 1. What the artifact looks like

### In plain terms

Look at a raw frame and three things stand out. First, the stripes are packed
more tightly at the left and right edges than in the middle. Second, if you
compare the pattern on any two rows, it is the same. Third, if you watch the
movie, the pattern drifts slowly sideways. Each of these is a clue about where
the stripes come from, and the removal method is built on all three.

### In detail

* The *column profile* of a frame is the mean over its rows, one value per
  column. Because the ripple is identical on every row, it survives this
  averaging while pixel noise shrinks by $\sqrt{256} = 16$.
* The *dynamic profile* is the column profile minus the temporal mean of the
  column profiles over the block. The real image also has column structure
  (vessels, gradients) but it does not move; the ripple does, so the temporal
  mean is a nearly ripple-free image and subtracting it leaves ripple, noise
  and activity.
* Even and odd rows give the same profile (correlation about 0.9); the top and
  bottom halves too (about 0.8).
* Stacking the dynamic profiles over time produces diagonal bands: the
  pattern advances by a fraction of a stripe per frame.

### Why?

* *Why not just look at pixels?* One column of one frame is dominated by
  photon noise (standard deviation about 50 counts versus a stripe amplitude
  of about 17). Averaging rows is what makes the pattern visible and
  measurable.
* *Why subtract the temporal mean instead of, say, a smoothed version of the
  frame?* Because the temporal mean is a physical ripple-free reference: the
  crawl guarantees that the ripple averages to zero over many frames, while
  everything static stays. No assumption about image smoothness is needed.
* *Why does the crawl matter?* It is both a nuisance and a gift. It means the
  pattern has to be re-estimated for every frame, but it is also what lets the
  temporal mean separate image from ripple.

In [ ]:
frame = mov[0]
prof = mov.mean(axis=1)                      # column profile of every frame: mean over rows, (T, W)
dyn = prof - prof.mean(axis=0)               # dynamic part: static image structure removed

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
vmin, vmax = np.percentile(frame, [1, 99])
axes[0, 0].imshow(frame, cmap="gray", vmin=vmin, vmax=vmax)
axes[0, 0].set_title("raw frame: vertical bands, tighter at the edges")
axes[0, 0].axis("off")

axes[0, 1].plot(dyn[0], lw=0.8)
axes[0, 1].set_xlabel("column")
axes[0, 1].set_ylabel("counts")
axes[0, 1].set_title("column profile of one frame (static image removed)")

fr = frame - frame.mean(axis=1, keepdims=True)
even, odd = fr[0::2].mean(axis=0), fr[1::2].mean(axis=0)
top, bottom = fr[: H // 2].mean(axis=0), fr[H // 2 :].mean(axis=0)
axes[1, 0].plot(even, lw=0.8, label=f"even rows")
axes[1, 0].plot(odd, lw=0.8, label=f"odd rows  (corr {np.corrcoef(even, odd)[0, 1]:.2f})")
axes[1, 0].plot(bottom - 40, lw=0.8, label=f"bottom half, offset  (corr top/bottom {np.corrcoef(top, bottom)[0, 1]:.2f})")
axes[1, 0].set_xlim(80, 176)
axes[1, 0].set_xlabel("column")
axes[1, 0].set_title("the same pattern on every row")
axes[1, 0].legend(fontsize=8)

lim = np.percentile(np.abs(dyn), 99)
axes[1, 1].imshow(dyn[:150], aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim, interpolation="nearest")
axes[1, 1].set_xlabel("column")
axes[1, 1].set_ylabel("frame")
axes[1, 1].set_title("column profiles over time: the pattern crawls")
fig.tight_layout()

## 2. Why the bands narrow at the edges: the resonant scanner

### In plain terms

The mirror that steers the laser across a line swings like a pendulum. A
pendulum moves fastest through the middle of its swing and almost stops at the
ends. The image is drawn with equally spaced pixels, so near the edges of the
line the beam spends more time on each pixel than in the middle. Now imagine a
hum that ticks at a perfectly steady rate. Where the beam moves slowly, more
ticks fit into one pixel width, so the stripes come out narrower. The
microscope software even records how long it spends on every pixel, so the
exact stretching of the time axis is known for each column.

### In detail

Let $\tau(x)$ be the time within a line at which column $x$ is recorded. A
ripple at frequency $f$ has phase $2\pi f \tau(x)$ along the line, so its
local spatial frequency is

$$ f_x(x) = f\,\frac{d\tau}{dx}, $$

proportional to the dwell time per pixel. With spatial fill fraction $F_s$,
scanner frequency $f_{res}$ and $u = F_s\,(2x/W - 1)$:

$$ \tau(x) = \frac{\arcsin u}{2\pi f_{res}} + \text{const}, \qquad
   \frac{d\tau}{dx} \propto \frac{1}{\sqrt{1-u^2}} . $$

ScanImage stores the number of ADC samples per pixel in `SI.hScan2D.mask`
(158 at the edges, 70 in the center here). `ScanGeometry` builds $\tau(x)$
from the mask when present and from the formula otherwise; they agree to
better than $10^{-4}$ of the line window. The figure below measures the local
stripe frequency in eight column windows and compares it with $d\tau/dx$ from
the metadata: the edge-to-center ratio comes out 1.62 measured versus 1.60
predicted.

### Why?

* *Why does this matter for removal?* A filter working in pixel coordinates
  sees a different stripe frequency at every position and cannot remove the
  pattern cleanly anywhere. Once columns are converted to time, a single
  frequency describes the ripple everywhere along the line.
* *Why trust the mask rather than the formula?* The mask is what the
  acquisition actually used, including rounding to whole samples; the formula
  is the fallback for files that lack it. Since they agree, either works.
* *Why is the ratio 1.6 and not the full 2.3 implied by the fill fraction?*
  The windows are 32 columns wide, so the outermost window is centered 16
  columns in from the edge, where the dwell is not yet at its maximum.

In [ ]:
mask = geometry.mask
tau = geometry.pixel_times()
tau_arcsin = geometry.replace(mask=None).pixel_times()
dt_dx = np.gradient(tau)                                   # seconds per pixel

# measured local stripe frequency in column windows
nwin, wlen = 8, W // 8
ff = np.fft.rfftfreq(4096)
meas = []
for w in range(nwin):
    seg = dyn[:, w * wlen:(w + 1) * wlen] * np.hanning(wlen)
    m = np.abs(np.fft.rfft(seg, n=4096, axis=1)).mean(axis=0)
    m[ff < 0.01] = 0
    meas.append(ff[np.argmax(m)])
meas = np.array(meas)
xc = (np.arange(nwin) + 0.5) * wlen
pred = dt_dx[xc.astype(int)]
center = slice(nwin // 2 - 1, nwin // 2 + 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(mask, lw=1)
axes[0].set_xlabel("column")
axes[0].set_ylabel("ADC samples per pixel")
axes[0].set_title("ADC samples per pixel (SI.hScan2D.mask)")
axes[1].plot(tau * 1e6, lw=1, label="from mask")
axes[1].plot((tau_arcsin - tau_arcsin[W // 2] + tau[W // 2]) * 1e6, "--", lw=1, label="arcsin formula")
axes[1].set_xlabel("column")
axes[1].set_ylabel("time within line (us)")
axes[1].set_title("acquisition time of each column")
axes[0].set_title("dwell per pixel (mask)")
axes[1].legend()
axes[2].plot(xc, meas / meas[center].mean(), "o-", label="measured stripe frequency")
axes[2].plot(xc, pred / pred[center].mean(), "s--", label="predicted from d(tau)/dx")
axes[2].set_xlabel("column (window center)")
axes[2].set_ylabel("local frequency / center")
axes[2].set_title("local stripe frequency vs prediction")
axes[2].legend()
fig.tight_layout()
print(f"edge / center frequency ratio: measured {meas[0] / meas[center].mean():.2f}, predicted {pred[0] / pred[center].mean():.2f}")

## 3. Look at the stripes in scan time

### In plain terms

Imagine stretching the image horizontally so that every pixel becomes as wide
as the time the beam spent on it. In that stretched picture the stripes become
perfectly regular, like the teeth of a comb. A frequency plot of the stretched
profile then shows a few sharp peaks: the base tone of the hum at 100 kHz and
its overtones at 200, 300, 400 kHz and so on, exactly like a musical note and
its harmonics. In the unstretched picture the same plot is a smeared blob,
because the stripe spacing keeps changing across the line.

### In detail

`tspace_spectrum` repeats each pixel value as many times as its dwell count,
which resamples the profile onto the uniform 125 MHz ADC clock (22,278 samples
per line window of 178 us). Each frame gets a Hann window and a zero-padded
FFT, and the squared magnitudes are averaged over frames. The lines are as
narrow as one line window allows (about 5.6 kHz full width) and up to eight
harmonics are visible. The strongest line sits at 100.1 kHz, which is 50.07
times the 2 kHz line rate; the exact index is settled in the next section.

### Why?

* *Why harmonics?* The hum is not a pure sine wave. Its shape (a fundamental
  with a strong 2nd and 4th harmonic) is typical of a switching power supply,
  which is the usual source of ripple in photomultiplier electronics. Every
  harmonic is a separate set of narrower stripes, so all of them have to be
  modelled; a one-frequency notch would leave most of the pattern behind.
* *Why the Hann window and zero padding?* The window keeps the strong lines
  from leaking into their neighbourhood; the padding interpolates the spectrum
  so peak positions and heights are read accurately. Neither changes the
  physics.
* *Why does the pixel-coordinate spectrum look smeared?* Because the local
  frequency ranges from about 0.056 to 0.13 cycles per pixel across the line;
  the power of one tone is spread over all of those bins.

In [ ]:
fx = np.fft.rfftfreq(W)
px = (np.abs(np.fft.rfft(dyn * np.hanning(W), axis=1)) ** 2).mean(axis=0)
ft, pt = tspace_spectrum(dyn, geometry)
sel = (ft > 5e3) & (ft < 950e3)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].semilogy(fx[1:], px[1:], lw=0.8)
axes[0].set_xlabel("spatial frequency (cycles / pixel)")
axes[0].set_ylabel("power")
axes[0].set_title("pixel coordinate: smeared")
axes[1].semilogy(ft[sel] / 1e3, pt[sel], lw=0.8)
axes[1].set_xlabel("frequency in scan time (kHz)")
axes[1].set_title("scan time: sharp harmonic lines")
fig.tight_layout()

f_peak = ft[sel][np.argmax(pt[sel])]
line_rate = 1 / geometry.line_period_s
print(f"strongest line at {f_peak / 1e3:.1f} kHz = {f_peak / line_rate:.2f} x the line rate ({line_rate:.0f} Hz)")

## 4. Why the bands are vertical, and why they crawl

### In plain terms

The scanner draws 2,000 lines per second and the hum ticks 100,000 times per
second, so there are exactly 50 ticks per line. Every new line therefore starts
at the same point of the hum, and the band pattern repeats identically line
after line: that is what makes the bands vertical. If it were 50.5 ticks per
line, every other line would show the pattern shifted by half a stripe and we
would see a checkerboard instead.

It is not *exactly* 50, only very nearly. The tiny mismatch adds up: after one
whole frame (256 image lines plus 10 lines of fly-back, 266 line times) the
pattern has moved by about 3 % of one stripe. That is the slow slide seen in
the movie. How large the mismatch is depends on the exact speed of the
scanner, which drifts as the microscope warms up, so it is measured from the
data rather than assumed.

### In detail

The fundamental sits at $f_0 = n / T_{line}$ with integer $n = 50$. The
residual phase advance per line, $\rho$ (cycles per row), has two visible
effects: a phase ramp of $\rho H$ cycles from the top to the bottom of a frame,
and a crawl of $\rho L$ cycles per frame, where $L = 266$ is the number of
line periods per frame including fly-back. On this block, $\rho \approx
1.3\times10^{-4}$ cycles per row, giving a ramp of 0.033 cycles per frame and
a crawl of 0.034 cycles per frame. `detect_ripple` estimates $\rho$ twice,
from the frame-to-frame phase of the fundamental and from the phase differences
between row blocks within single frames; the two agree to a few percent, which
confirms the geometry (the factor 266 in particular).

### Why?

* *Why track $\rho$ instead of fitting it once?* Across the six files of the
  example session the crawl rate falls steadily from file to file, and it
  changes within a file too. A fixed value would be wrong within minutes; the
  per-frame phase track follows it for free.
* *Why 266 and not 256 line periods per frame?* The scanner keeps running
  during the 5 ms fly-back between frames, so the hum advances during it as
  well. Using 256 would give a crawl prediction 4 % too small, and the
  agreement between the two $\rho$ estimates would be lost.
* *Why does the histogram of per-frame advances have some width?* Partly
  measurement noise on the phase (about 0.003 cycles per frame) and partly
  genuine short-term variation of the scanner. This is why the phase is
  estimated for every frame instead of extrapolated.

In [ ]:
det = detect_ripple(mov, geometry)
print(f"detected: {det.detected}   harmonic index n = {det.harmonic_index}   f0 = {det.f0_hz / 1e3:.3f} kHz   SNR {det.snr_db:.1f} dB")
print(f"harmonics visible: {det.n_harmonics_detected}   fundamental amplitude {det.amplitude_counts:.1f} counts")
print(f"phase advance per frame {det.phase_advance_cycles_per_frame:.4f} cycles  ->  rho = {det.rho_cycles_per_row:.2e} cycles/row")
print(f"rho from row blocks within a frame: {det.rho_row_block:.2e} cycles/row   (ramp over one frame: {det.rho_cycles_per_row * H:.3f} cycles)")

# a one-harmonic fit per frame gives the phase track of the fundamental
groups = RowGroups.from_geometry(geometry, 8)
profiles = column_profiles(mov, groups)
fit1 = fit_ripple(profiles, groups, geometry, det.f0_hz, n_harmonics=1)
phase = np.unwrap(2 * np.pi * fit1.phase[:, 0]) / (2 * np.pi)
adv = fit1.phase_advance()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(phase, lw=1)
axes[0].set_xlabel("frame")
axes[0].set_ylabel("fundamental phase (cycles)")
axes[0].set_title("the ripple phase advances steadily: the crawl")
axes[1].hist(adv, bins=40)
axes[1].axvline(np.median(adv), color="r", ls="--", label=f"median {np.median(adv):.4f} cycles/frame")
axes[1].set_xlabel("phase advance per frame (cycles)")
axes[1].set_title(f"= rho x {geometry.lines_per_frame_period} line periods")
axes[1].legend()
fig.tight_layout()

## 5. The model and the fit

### In plain terms

At this point the shape of the pattern is known in every detail except, for
each frame and each overtone, two numbers: how strong it is and where its
peaks sit (its phase). Finding them is like sliding a stencil of known shape
over the frame until it matches best, then noting the size and position that
gave the best match. "Best" means the smallest total squared difference
between the stencil and the data, which has a direct, exact solution. Each
frame gets its own stencil setting because the pattern slides.

### In detail

For frame $t$, row $r$, column $x$ and harmonics $k = 1 \dots K$:

$$ \text{ripple}(t, r, x) = \sum_{k=1}^{K} A_k(t)\, g_k(x)\,
   \cos\!\Big( 2\pi k \big[ f_0\,\tau(x) + \rho_t\,(r - r_0) \big] + \varphi_k(t) \Big) $$

* $\tau(x)$ and $g_k(x)$ come from the scan geometry; $g_k$ is the
  attenuation of harmonic $k$ caused by averaging each pixel over its dwell
  time (a sinc-like factor). It is a few percent for the fundamental, about
  40 % for the 4th harmonic at the edges, and nearly complete for the 8th at
  the extreme edges, where one pixel (1.26 us) spans a whole 800 kHz cycle.
  Without this factor the fit would over-subtract high harmonics at the edges.
* $f_0 = n / T_{line}$ from detection; $\rho_t$ from the phase track.
* The unknowns per frame are $A_k(t)$ and $\varphi_k(t)$, or equivalently one
  complex coefficient $c_k(t)$ per harmonic. Writing the cosine as a cosine
  and a sine term makes the model linear in the unknowns, so they are found
  by ordinary least squares: a design matrix of 8 row groups x 256 columns =
  2048 rows and $2K = 16$ columns, solved for 256 frames at a time.
* Each row group enters with its own phase factor $e^{2\pi i k \rho (r_g - r_0)}$
  and a small correction for averaging the ramp within the group.
* The static image structure is the temporal mean of the profiles; it is
  subtracted before fitting and re-estimated once from the residual (as a
  running mean over 1024 frames) so that an incomplete crawl cycle or slow
  image drift does not bias it.

The figure shows the fundamental basis function (a chirp: tighter at the
edges), the measured dynamic profile of one frame against the fitted model,
the residual, and the median amplitude of each harmonic. The residual standard
deviation (about 3.8 counts) matches the photon-noise floor of a whole-frame
column profile (about 54 / $\sqrt{256}$ = 3.4 counts), i.e. nothing systematic
is left.

### Why?

* *Why is the explained variance "only" 0.67?* The dynamic profiles contain
  ripple, photon noise and residual activity. The ripple part is removed
  completely (section 6 shows the lines at the noise floor); the rest is not
  ripple and must not be removed. The number is a property of the data, not a
  shortfall of the fit.
* *Why eight harmonics?* The 4th harmonic alone carries 40 % of the
  fundamental's amplitude and lines are visible up to the 8th. With four
  harmonics the higher lines remain about 50 times above background; with
  eight they are at the floor. More than eight adds parameters without
  benefit on this data.
* *Why weight out the brightest pixels?* Active cells change brightness from
  frame to frame, and that change has column structure that could be mistaken
  for ripple. The ripple is present everywhere, so excluding 5 % of pixels
  costs nothing.
* *Why re-estimate the static profile?* The first estimate is a plain temporal
  mean, which still contains a small amount of ripple if the block does not
  cover whole crawl cycles. Subtracting the first ripple fit and averaging
  again removes that bias.

In [ ]:
fit = fit_ripple(profiles, groups, geometry, det.f0_hz, n_harmonics=8)
print(f"explained variance of the dynamic profiles: {fit.explained_variance:.2f}")
print("median amplitude per harmonic (counts):", np.round(np.median(fit.amplitude[:, 0, :], axis=0), 1))

# the design: cos/sin of each harmonic in scan time (fundamental shown)
basis = ripple_basis(geometry, fit.f_hz, 8)[0]
t = 100                                                   # frame to show
g_mean = profiles[:, :, :].mean(axis=1)                   # (T, W) whole-frame profile
static = (profiles - fit_model if False else None)         # placeholder to keep the cell self-contained
from bandaid.ripple import _model_on_groups                 # model on the same row-group grid as the fit
model_g = _model_on_groups(fit.coeffs, fit.rho, fit.basis, fit.groups)
resid_g = profiles - model_g
static_g = resid_g.mean(axis=0)
measured = (profiles[t] - static_g).mean(axis=0)
measured = measured - measured.mean()                    # the model has no DC term; a frame's overall brightness is not ripple
modelled = model_g[t].mean(axis=0)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
axes[0, 0].plot(basis[0].real, lw=0.8, label="k = 1")
axes[0, 0].plot(basis[1].real, lw=0.6, alpha=0.6, label="k = 2")
axes[0, 0].set_xlim(0, 128)
axes[0, 0].legend(fontsize=8)
axes[0, 0].set_title("basis: cos(2 pi k f0 tau(x)) is a chirp in x (left half)")
axes[0, 0].set_xlabel("column")
axes[0, 1].plot(measured, lw=0.8, label="measured dynamic profile")
axes[0, 1].plot(modelled, lw=0.8, label="fitted 8-harmonic model")
axes[0, 1].set_xlim(0, 128)
axes[0, 1].set_title(f"frame {t}: measured vs 8-harmonic model (left half)")
axes[0, 1].legend(fontsize=8)
axes[1, 0].plot(measured - modelled, lw=0.8, color="0.3")
axes[1, 0].set_title(f"residual: std {np.std(measured - modelled):.2f} vs stripe std {np.std(measured):.2f} counts")
axes[1, 0].set_xlabel("column")
axes[1, 1].bar(np.arange(1, 9), np.median(fit.amplitude[:, 0, :], axis=0))
axes[1, 1].set_xlabel("harmonic k")
axes[1, 1].set_ylabel("median amplitude (counts)")
axes[1, 1].set_title("harmonic content (K > 1 is needed)")
fig.tight_layout()

## 6. Subtract, then check

### In plain terms

The fitted pattern is erased from every pixel of every frame. Then three
things are verified. In the stretched-picture frequency plot, the peaks must be
gone. The average image over time must be unchanged, because the sliding
stripes average out anyway, so if the average moved we would have removed
something real. And the part that was removed, viewed as an image, must look
like stripes and nothing else.

### In detail

The model is evaluated on the full frame grid (each column with its phase from
$\tau(x)$, each row with the ramp, each harmonic with its attenuation), which
is a small matrix product per frame, and subtracted. Integer output is rounded
to nearest and clipped to the dtype range. The checks:

1. Line power over local background at each harmonic of the residual
   profiles: before, 12 to 1000; after, below 0.5 for all eight.
2. Largest change of the temporal-mean image: 0.009 counts on this block.
3. The removed component is shown as an image.

`remove_ripple` runs all of this in one call and also streams whole files to
disk; here it is applied to the in-memory block with the same settings, and
agrees with the manual steps to within one count (the rounding).

### Why?

* *Why are the line ratios slightly below 1 after cleaning?* Least squares
  removes whatever sits exactly at the harmonic frequencies, including the
  little bit of noise that happens to be there, so the spectrum dips slightly
  at those bins. It is the signature of a complete removal, not of damage:
  the dip is a few bins wide and holds a negligible amount of noise power.
* *Why is the mean image such a strong test?* Any error in the model that is
  not itself crawling would accumulate in the mean. Real ripple averages out
  over about 30 frames; a bias of even one count would be visible here.
* *Why round and clip rather than truncate?* Truncation toward zero biases
  values by half a count and, for unsigned types, a value pushed below zero
  would wrap to 65535. Rounding and clipping keep the output faithful.

In [ ]:
model = ripple_frames(fit, range(T))
cleaned = mov - model

groups1 = RowGroups.from_geometry(geometry, 1)
before = column_profiles(mov, groups1)[:, 0]
after = column_profiles(cleaned, groups1)[:, 0]
ratios_before = line_power_ratios(before - before.mean(0), geometry, fit.f0_hz, 8)
ratios_after = line_power_ratios(after - after.mean(0), geometry, fit.f0_hz, 8)
print("line power / background before:", {k: round(v) for k, v in ratios_before.items()})
print("line power / background after: ", {k: round(v, 2) for k, v in ratios_after.items()})
print(f"largest change of the temporal-mean image: {np.abs(model.mean(axis=0)).max():.3f} counts")

fig, ax = plt.subplots(figsize=(11, 4))
plot_tspace_spectra(before, after, geometry, fit.f0_hz, 8, ax=ax)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.3))
vmin, vmax = np.percentile(mov[t], [1, 99])
axes[0].imshow(mov[t], cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title("raw")
axes[1].imshow(cleaned[t], cmap="gray", vmin=vmin, vmax=vmax)
axes[1].set_title("cleaned")
lim = np.percentile(np.abs(model[t]), 99)
axes[2].imshow(model[t], cmap="RdBu_r", vmin=-lim, vmax=lim)
axes[2].set_title("removed: stripes only, tighter at the edges")
for ax in axes:
    ax.axis("off")
fig.tight_layout()

# the one-call API (same parameters; the default fit_mask='auto' additionally ignores the brightest pixels)
cleaned_api, report = remove_ripple(mov_int, geometry, n_harmonics=8, fit_mask=None)
print(f"one-call API agrees with the manual steps to {np.abs(cleaned_api.astype(np.float32) - np.rint(cleaned)).max():.0f} count(s)")

## 7. Additive, not gain modulation

### In plain terms

A hum can enter a recording in two ways. It can simply add to the signal, like
background noise, or it can wobble the detector's amplification so that
bright things wobble more than dark things. The two need different fixes:
subtract in the first case, divide in the second. Two tests settle it: turn
the laser up from 50 % to 90 % and compare the stripe amplitude, and compare
bright and dark rows within one frame. The amplitude does not change in either
test, so the hum is added, and subtracting it is right.

### In detail

The fundamental amplitude is 16.8 counts at 50 % laser power and 17.2 at 90 %
(mean intensities 36 and 40). Within a frame, rows sorted into brightness
quartiles all show about 17.5 counts, and the correlation between row
brightness and stripe amplitude is zero within noise. A gain modulation would
scale the amplitude with the signal.

### Why?

* *Why does it matter?* If the ripple were multiplicative, subtracting a fixed
  pattern would under-correct bright cells and over-correct dark background;
  the model would have to be applied as a divisive normalisation instead.
* *Why do the two tests agree with a power-supply origin?* Ripple on a
  photomultiplier's output stage or on the amplifier adds a signal-independent
  voltage; ripple on the high-voltage supply would change the gain and would
  scale with brightness.

In [ ]:
path90 = data_dir / "HK13_20260814_90pw_00006.tif"
mov90 = load_tiff_stack(path90, frames=range(2000, 2300))
det90 = detect_ripple(mov90, geometry)
print(f"fundamental amplitude: 50% power {det.amplitude_counts:.1f} counts, 90% power {det90.amplitude_counts:.1f} counts "
      f"(mean intensity {mov.mean():.1f} vs {mov90.mean():.1f})")

# per-row amplitude of the fundamental vs row brightness, on a few frames
D = np.concatenate([basis[:1].real.T, -basis[:1].imag.T], axis=1)          # (W, 2)
row_static = (mov - mov.mean(axis=2, keepdims=True)).mean(axis=0)          # static row-demeaned image
amps, bright = [], []
for k in range(0, T, 20):
    fr = mov[k] - mov[k].mean(axis=1, keepdims=True) - row_static
    c, *_ = np.linalg.lstsq(D, fr.T, rcond=None)
    amps.append(np.hypot(c[0], c[1]))
    bright.append(mov[k].mean(axis=1))
amps, bright = np.concatenate(amps), np.concatenate(bright)
q = np.quantile(bright, [0, 0.25, 0.5, 0.75, 1])
for lo, hi in zip(q[:-1], q[1:]):
    m = (bright >= lo) & (bright <= hi)
    print(f"rows with mean intensity {lo:6.1f}..{hi:6.1f}: stripe amplitude {amps[m].mean():.1f} counts")
print(f"correlation(row brightness, stripe amplitude) = {np.corrcoef(bright, amps)[0, 1]:.2f}")

## 8. How detection decides

### In plain terms

Before touching a file, the software checks that the hum is really there,
with three independent tests: there must be a clear peak in the stretched
frequency plot, that peak must stand well above its immediate neighbourhood,
and a single-tone version of the stencil must explain a meaningful share of
the stripe variation. All three have to pass; otherwise the file is left
untouched and the report says so.

### In detail

`detect_ripple` uses three blocks of consecutive frames (start, middle, end)
and requires:

* **SNR** $\geq$ 10 dB: the strongest line in the search band is at least ten
  times the median power of the band;
* **line ratio** $\geq$ 10 at the fundamental: peak power within half a line
  width of $f_0$ over the median power three to ten line widths away;
* **explained variance** $\geq$ 0.1 for a one-harmonic fit.

The harmonic index $n$ is not read off the peak alone: one line window is only
178 us long, so the spectral resolution (5.6 kHz) is coarser than the 2 kHz
spacing between neighbouring multiples of the line rate. Instead, the integers
around the peak are tried and the one whose fit explains the most variance
wins. On this block the raw data score 30 dB, a line ratio near 1000 and an
explained variance of 0.58; the cleaned block scores 1.4 dB, 1.2 and 0.01 and
is correctly rejected.

### Why?

* *Why three criteria?* Each can be fooled alone: a strongly coloured noise
  floor can produce a modest SNR without any line, a line ratio can be high
  for a weak but very narrow interference, and explained variance can be
  inflated by structured activity. Together they have not produced a false
  positive on clean data.
* *Why decide $n$ by fit rather than by peak position?* Because an error of
  one in $n$ shifts the model phase by up to a third of a cycle at the line
  edges and ruins the fit, while the spectral peak cannot distinguish 49, 50
  and 51.
* *Why not just always run the removal?* On a file without ripple the fit
  would subtract a small amount of noise-shaped pattern for no benefit and
  would report misleading metrics. Refusing is the safer default; `force`
  overrides it.

In [ ]:
det_clean = detect_ripple(cleaned_api, geometry)
for label, d in [("raw block", det), ("cleaned block", det_clean)]:
    print(f"{label:14s} detected={d.detected!s:5s} SNR {d.snr_db:5.1f} dB   line ratio k=1 {d.line_power_ratio[1]:8.1f}   "
          f"EV(1 harmonic) {d.explained_variance_k1:.2f}   notes: {d.notes}")

## 9. Assumptions and limits

### In plain terms

The method needs to know how the microscope draws its lines; it assumes the
hum repeats almost exactly a whole number of times per line, which is exactly
the situation that produces vertical bands; and it assumes the hum adds to the
signal rather than scaling it. Within those conditions it follows changes of
the hum from frame to frame. Scanning in both directions (left to right and
right to left on alternate lines) is supported in the code but has not been
tried on real data yet.

### In detail

* **Scan geometry is required.** It comes from the ScanImage metadata; for
  other systems a `ScanGeometry` can be built from the fill fraction and
  scanner frequency, or a uniform one for galvo scanning.
* **The ripple must be close to a multiple of the line rate.** That is what
  makes the bands vertical in the first place; a ripple far from a multiple
  would show up as diagonal patterns and would average out of the row
  profiles, so the detector would (correctly) not fire.
* **Additive**, as verified above for this system.
* **Per-frame tracking.** Amplitude and phase are fitted for every frame, so
  changes over a recording are followed; the crawl rate is smoothed over
  about 50 frames, so it must not change faster than that.
* **Bidirectional scanning** is implemented (even and odd lines are fitted
  separately) but has not yet been validated on real data.
* **What to look at if it fails:** the diagnostics figures
  (`bandaid-clean --diagnostics`): remaining lines in the scan-time spectrum,
  a noisy phase track, or cell structure visible in the removed component.

## Glossary and frequently asked questions

**Line rate, line period.** Lines drawn per second (2,000 here) and the time
between the starts of two lines (0.5 ms). A frame of 256 lines plus 10 lines
of fly-back takes 266 line periods, 0.133 s.

**Fill fraction, dwell, mask.** Only the central part of the mirror swing is
used for imaging (spatial fill fraction 0.9). The dwell is the time spent on
one pixel; the mask lists it for every column in ADC samples.

**Scan time.** The time axis along a line. Converting columns to scan time
undoes the pendulum-like stretching and makes the ripple regular.

**Harmonic index $n$.** How many ripple cycles fit in one line period (50).
**Harmonic $k$.** Overtone number: $k = 1$ is the fundamental at 100 kHz,
$k = 2$ is 200 kHz, and so on.

**Phase.** Where in its cycle the ripple is at a given moment, in cycles
(0 to 1). **Crawl rate $\rho$.** How much the phase advances per image row
because the ripple frequency is not exactly $n$ per line; $\rho$ times 266 is
the advance per frame.

**Column profile.** The average of a frame's rows, one number per column.
**Dynamic profile.** The column profile minus its temporal mean over frames.

**Least squares.** Choosing the model parameters that minimise the summed
squared difference between model and data; for a linear model it has a
direct solution. **Explained variance.** The fraction of the data's variance
accounted for by the model.

**SNR (here).** Ten times the log of the strongest spectral peak over the
median power of the search band, in dB. A relative detection statistic, not
an amplitude ratio. **Line ratio.** Peak power at a harmonic over the median
power in a local annulus around it.

---

**Does this remove real signal?** The model can only produce the known
stripe shape at the known frequencies, and the temporal-mean image changes by
less than a tenth of a count. Cell activity is not at those frequencies and is
excluded from the fit by the mask.

**What if the ripple frequency or strength changes during a recording?** The
amplitude and phase are fitted for every frame, and the crawl rate is tracked;
only the harmonic index is fixed per file. A change of the harmonic index
itself (a different power supply frequency) would need a new detection.

**Does it work on other microscopes?** Any resonant-scanner system whose
metadata gives the fill fraction and scanner frequency (or a dwell mask) can
be handled; galvo systems with uniform pixel timing use a uniform geometry.
The ripple must still be locked to the line rate for the bands to be vertical.

**How long does it take?** About 6 s per 4512-frame file from local disk and
under 1 GB of memory, independent of movie length; about 60 s per file over a
network share, dominated by reading the file twice.

**What is in the report?** Per file: the detection numbers, the fit summary
(amplitudes, crawl rate, outlier frames), the line ratios before and after,
the mean-image change, and the run time. It is written next to the cleaned
files as `ripple_report.json`.